# Multi-Modal IR — Exploratory Data Analysis & Evaluation

This notebook covers:
1. Corpus statistics (papers, figures, abstract length distribution)
2. Embedding space visualization (PCA/UMAP of text and figure embeddings)
3. Evaluation results visualization (MRR, NDCG, Recall@k)
4. Fusion weight heatmap (from calibration grid search)
5. Cross-modal retrieval examples

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from src.db import get_session, Paper, Figure, init_db
from src.config import cfg

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Corpus Statistics

In [ ]:
init_db()
session = get_session()
papers  = session.query(Paper).all()
figures = session.query(Figure).all()
session.close()

print(f'Papers in corpus:  {len(papers)}')
print(f'Figures extracted: {len(figures)}')
print(f'Avg figures/paper: {len(figures)/max(len(papers),1):.2f}')

# Abstract length distribution
abs_lengths = [len((p.abstract or '').split()) for p in papers]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(abs_lengths, bins=30, color='#6c5ce7', edgecolor='white')
axes[0].set_title('Abstract Length Distribution (words)')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Papers')

figs_per_paper = pd.Series([p.paper_id for p in figures]).value_counts()
axes[1].hist(figs_per_paper.values, bins=15, color='#00b894', edgecolor='white')
axes[1].set_title('Figures per Paper Distribution')
axes[1].set_xlabel('Number of figures')
axes[1].set_ylabel('Papers')

plt.tight_layout()
plt.savefig('../data/processed/corpus_stats.png', bbox_inches='tight')
plt.show()

## 2. Embedding Space Visualization (PCA)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

text_emb_path = Path('../data/indices/text_embeddings.npy')
if text_emb_path.exists():
    text_vecs = np.load(text_emb_path)
    print(f'Text embeddings shape: {text_vecs.shape}')

    # Reduce to 2D
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(text_vecs[:500])  # Use up to 500 papers

    # Color by year if available
    years = [p.year or 2020 for p in papers[:500]]

    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=years,
                          cmap='plasma', alpha=0.6, s=20)
    plt.colorbar(scatter, label='Publication Year')
    plt.title('PCA of Paper Abstract Embeddings (SBERT, 768-d → 2-d)')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
    plt.savefig('../data/processed/embedding_pca.png', bbox_inches='tight')
    plt.show()
else:
    print('Run build_index.py first to generate embeddings')

## 3. Evaluation Results

In [ ]:
eval_path = Path('../data/eval_results.json')
if eval_path.exists():
    with open(eval_path) as f:
        eval_data = json.load(f)

    hybrid = eval_data.get('hybrid', {})
    k_vals = sorted([int(k) for k in hybrid.get('ndcg', {}).keys()])

    metrics = {
        'MRR':       [hybrid['mrr'].get(str(k), 0) for k in k_vals],
        'NDCG':      [hybrid['ndcg'].get(str(k), 0) for k in k_vals],
        'Recall':    [hybrid['recall'].get(str(k), 0) for k in k_vals],
        'Precision': [hybrid['precision'].get(str(k), 0) for k in k_vals],
    }

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#6c5ce7', '#00b894', '#e17055', '#fdcb6e']
    for (metric, values), color in zip(metrics.items(), colors):
        ax.plot(k_vals, values, marker='o', label=metric, color=color, linewidth=2)

    ax.set_xticks(k_vals)
    ax.set_xlabel('Cut-off k')
    ax.set_ylabel('Score')
    ax.set_title('IR Evaluation Metrics (Hybrid SBERT+CLIP+BM25)')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('../data/processed/eval_metrics.png', bbox_inches='tight')
    plt.show()
else:
    print('Run scripts/evaluate.py first')

## 4. Fusion Weight Grid Search Heatmap

In [ ]:
calib_path = Path('../configs/calibrated_weights.json')
if calib_path.exists():
    with open(calib_path) as f:
        calib = json.load(f)

    for qtype, res in calib.items():
        grid = res.get('grid', [])
        if not grid:
            continue

        df = pd.DataFrame(grid)
        pivot = df.pivot(index='beta', columns='alpha', values='score')

        plt.figure(figsize=(9, 7))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis',
                    linewidths=0.3, cbar_kws={'label': 'NDCG@10'})
        plt.title(f'Fusion Weight Grid Search — {qtype} queries\n'
                  f'Best: α={res["best_alpha"]}, β={res["best_beta"]}')
        plt.xlabel('Text weight (α)')
        plt.ylabel('Figure weight (β)')
        plt.tight_layout()
        plt.savefig(f'../data/processed/fusion_heatmap_{qtype}.png', bbox_inches='tight')
        plt.show()
else:
    print('Run scripts/calibrate_fusion.py first')

## 5. Cross-Modal Retrieval Demo

In [ ]:
from src.fusion.search_engine import SearchEngine

engine = SearchEngine.get()
engine.load()

# Example cross-modal query
queries = [
    "U-shaped validation loss curve over training epochs",
    "attention heatmap visualization over tokens",
    "bar chart comparing BLEU scores across models",
]

for q in queries:
    print(f'\nQuery: "{q}"')
    resp = engine.search(text=q, top_k=3)
    print(f'  Query type: {resp.query_type}')
    print(f'  Latency: {resp.latency_ms} ms')
    for r in resp.results:
        print(f'  #{r.rank}: {r.paper["title"][:60]}...')
        print(f'       fused={r.fused_score:.4f}  text={r.text_score:.4f}  fig={r.figure_score:.4f}')
        if r.matched_figures:
            print(f'       Matched figure: {r.matched_figures[0]["caption"][:60]}')